# Data Validation & Pipeline Diagnostics

Run this notebook to diagnose data quality issues across the pipeline.

**Checks performed:**
1. Street distribution - where does postflop disappear?
2. Duplicate detection - where do duplicates first appear?
3. Action type distribution - are folds present?
4. Column inventory - what's missing at each stage?
5. Betting sanity checks:
   - Negative amounts
   - Impossible raises (amount < facing_call)
   - Same player acting twice consecutively
   - Raising after facing all-in (only call/fold valid)
   - Player betting after own all-in (impossible)
6. Trace single hand through pipeline
7. Compare raw actions to SP-2 output
8. Street ordering validation - is idx ordered correctly?
9. Pot progression validation - does pot grow correctly?
10. Action sequence validation - does the hand play out logically?
11. **NEW: Betting when facing shove** - deep dive with raw data comparison

In [ ]:
# Setup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("PIPELINE DATA VALIDATION")
print("=" * 80)

# Paths to check
PATHS = {
    'raw_actions': '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/chunks/phh_multi_actions_chunk_*.parquet',
    'sp2_output': '/Volumes/pokerml/default/data/processed/sp2_player_events_labeled',
    'sp3_output': '/Volumes/pokerml/default/data/processed/sp3_opponent_predictions',
    'sp4_output': '/Volumes/pokerml/default/data/processed/sp4_features_complete',
    'sp6_output': '/Volumes/pokerml/default/data/processed/sp6_features_enhanced',
    'sp7_output': '/Volumes/pokerml/default/data/processed/sp7_action_labels',
}

In [ ]:
# =============================================================================
# TEST 1: STREET DISTRIBUTION
# Where does flop/turn/river data disappear?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 1: STREET DISTRIBUTION")
print("=" * 80)

for name, path in PATHS.items():
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        total = df.count()
        print(f"Total rows: {total:,}")
        
        if 'street' in df.columns:
            print("\nStreet distribution:")
            street_dist = df.groupBy('street').count().orderBy('count', ascending=False)
            street_dist.show()
            
            # Calculate percentages
            for row in street_dist.collect():
                pct = 100 * row['count'] / total
                print(f"   {row['street']}: {row['count']:,} ({pct:.1f}%)")
        else:
            print("   WARNING: 'street' column not found!")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 2: DUPLICATE DETECTION
# Where do duplicates first appear?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 2: DUPLICATE DETECTION")
print("=" * 80)

for name, path in PATHS.items():
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        total = df.count()
        
        # Determine primary key columns
        if 'idx' in df.columns:
            pk_cols = ['hand_id', 'actor', 'idx']
        elif 'action_no_in_hand' in df.columns:
            pk_cols = ['hand_id', 'actor', 'action_no_in_hand']
        else:
            pk_cols = ['hand_id', 'actor', 'street', 'action_type']
        
        # Check for columns that exist
        pk_cols = [c for c in pk_cols if c in df.columns]
        
        if pk_cols:
            unique = df.select(pk_cols).distinct().count()
            duplicates = total - unique
            
            print(f"   Primary key: {pk_cols}")
            print(f"   Total rows: {total:,}")
            print(f"   Unique rows: {unique:,}")
            print(f"   Duplicates: {duplicates:,}")
            
            if duplicates > 0:
                print(f"   *** WARNING: {duplicates:,} DUPLICATE ROWS ***")
                
                # Show example duplicates
                print("\n   Example duplicates:")
                dup_keys = df.groupBy(pk_cols).count().filter(F.col('count') > 1).limit(3)
                dup_keys.show(truncate=False)
            else:
                print("   ✓ No duplicates")
        else:
            print("   WARNING: Could not determine primary key columns")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 3: ACTION TYPE DISTRIBUTION
# Are folds present? What actions exist?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 3: ACTION TYPE DISTRIBUTION")
print("=" * 80)

for name, path in PATHS.items():
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        
        if 'action_type' in df.columns:
            print("\nAction type distribution:")
            df.groupBy('action_type').count().orderBy('count', ascending=False).show()
            
            # Check for folds
            fold_count = df.filter(F.col('action_type') == 'fold').count()
            total = df.count()
            if fold_count == 0:
                print("   *** WARNING: NO FOLD ACTIONS ***")
            else:
                print(f"   Folds: {fold_count:,} ({100*fold_count/total:.1f}%)")
        else:
            print("   'action_type' column not found")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 4: COLUMN INVENTORY
# What columns exist at each stage? What's missing?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 4: COLUMN INVENTORY")
print("=" * 80)

# Critical columns to check
CRITICAL_COLS = [
    'hand_id', 'actor', 'street', 'action_type', 'amount',
    'idx', 'action_no_in_hand',  # Sequence columns
    'hole_cards', 'hole_cards_raw',  # Card columns
    'hand_equity', 'hand_strength',  # Strength columns
    'predicted_strength', 'predicted_bucket',  # Prediction columns
    'target_profit_bb', 'is_winner',  # Profit columns
    'position_from_button', 'position_name', 'position_bucket',  # Position columns
    'facing_call', 'pot_before_action', 'spr',  # Betting context
]

results = {}

for name, path in PATHS.items():
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        cols = set(df.columns)
        results[name] = cols
        
        print(f"   Total columns: {len(cols)}")
        print("\n   Critical columns:")
        for col in CRITICAL_COLS:
            status = '✓' if col in cols else 'MISSING'
            print(f"      {col}: {status}")
            
    except Exception as e:
        print(f"   ERROR: {e}")
        results[name] = set()

In [ ]:
# =============================================================================
# TEST 5: BETTING SANITY CHECKS
# Are betting values physically possible?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 5: BETTING SANITY CHECKS")
print("=" * 80)

from pyspark.sql.window import Window

for name, path in PATHS.items():
    if 'sp4' not in name and 'sp6' not in name and 'sp7' not in name and 'raw' not in name:
        continue
        
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        
        # Determine index column
        idx_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
        
        # Check 1: Negative amounts
        if 'amount' in df.columns:
            neg_amounts = df.filter(F.col('amount') < 0).count()
            print(f"   Negative amounts: {neg_amounts}")
            if neg_amounts > 0:
                print("   *** WARNING: Negative bet amounts ***")
        
        # Check 2: Raise amount < facing_call (impossible)
        if 'amount' in df.columns and 'facing_call' in df.columns and 'action_type' in df.columns:
            impossible_raises = df.filter(
                (F.col('action_type') == 'bet_or_raise_to') &
                (F.col('amount') < F.col('facing_call')) &
                (F.col('amount') > 0)
            ).count()
            print(f"   Impossible raises (amount < facing_call): {impossible_raises}")
            if impossible_raises > 0:
                print("   *** WARNING: Raises where amount < facing_call ***")
                df.filter(
                    (F.col('action_type') == 'bet_or_raise_to') &
                    (F.col('amount') < F.col('facing_call')) &
                    (F.col('amount') > 0)
                ).select('hand_id', 'actor', 'street', 'action_type', 'amount', 'facing_call').show(5)
        
        # Check 3: Same player acting twice consecutively
        if idx_col:
            w = Window.partitionBy('hand_id').orderBy(idx_col)
            
            df_with_prev = df.withColumn('prev_actor', F.lag('actor').over(w))
            consecutive = df_with_prev.filter(
                (F.col('actor') == F.col('prev_actor')) &
                (F.col('actor').isNotNull())
            ).count()
            
            print(f"   Consecutive actions by same player: {consecutive}")
            if consecutive > 0:
                print("   *** WARNING: Same player acting twice in a row ***")
                df_with_prev.filter(
                    (F.col('actor') == F.col('prev_actor')) &
                    (F.col('actor').isNotNull())
                ).select('hand_id', idx_col, 'actor', 'street', 'action_type', 'amount').show(5)
        
        # Check 4: ACTION AFTER ALL-IN (betting when facing a shove)
        # If previous action was all-in (amount == starting_stack), current player can only call/fold
        if idx_col and 'amount' in df.columns and 'starting_stack' in df.columns:
            w = Window.partitionBy('hand_id').orderBy(idx_col)
            
            df_with_context = df.withColumn('prev_amount', F.lag('amount').over(w)) \
                               .withColumn('prev_stack', F.lag('starting_stack').over(w)) \
                               .withColumn('prev_action', F.lag('action_type').over(w)) \
                               .withColumn('prev_actor', F.lag('actor').over(w))
            
            # Previous player went all-in (bet their entire stack)
            # Current player is raising (not just calling) - this is suspicious if facing all-in
            betting_after_allin = df_with_context.filter(
                (F.col('prev_amount') >= F.col('prev_stack') * 0.95) &  # Previous was ~all-in
                (F.col('prev_action') == 'bet_or_raise_to') &
                (F.col('action_type') == 'bet_or_raise_to') &
                (F.col('actor') != F.col('prev_actor'))  # Different player
            ).count()
            
            print(f"   Raises after opponent all-in: {betting_after_allin}")
            if betting_after_allin > 0:
                print("   *** WARNING: Player raising after facing all-in (only call/fold valid) ***")
                df_with_context.filter(
                    (F.col('prev_amount') >= F.col('prev_stack') * 0.95) &
                    (F.col('prev_action') == 'bet_or_raise_to') &
                    (F.col('action_type') == 'bet_or_raise_to') &
                    (F.col('actor') != F.col('prev_actor'))
                ).select(
                    'hand_id', idx_col, 'actor', 'action_type', 'amount', 
                    'prev_actor', 'prev_action', 'prev_amount', 'prev_stack'
                ).show(5, truncate=False)
        
        # Check 5: SAME PLAYER ALL-IN TWICE
        # A player who goes all-in cannot bet again in same hand
        if idx_col and 'amount' in df.columns and 'starting_stack' in df.columns:
            w = Window.partitionBy('hand_id', 'actor').orderBy(idx_col)
            
            df_player = df.withColumn('is_allin', 
                F.when(F.col('amount') >= F.col('starting_stack') * 0.95, 1).otherwise(0)
            ).withColumn('prev_allin', F.lag('is_allin').over(w))
            
            # Player betting after they already went all-in
            allin_then_bet = df_player.filter(
                (F.col('prev_allin') == 1) &
                (F.col('action_type') == 'bet_or_raise_to')
            ).count()
            
            print(f"   Player betting after own all-in: {allin_then_bet}")
            if allin_then_bet > 0:
                print("   *** CRITICAL: Player betting AFTER going all-in (impossible!) ***")
                df_player.filter(
                    (F.col('prev_allin') == 1) &
                    (F.col('action_type') == 'bet_or_raise_to')
                ).select('hand_id', idx_col, 'actor', 'street', 'action_type', 'amount', 'starting_stack').show(5)
                
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 6: TRACE A SINGLE HAND THROUGH PIPELINE
# Follow one hand from raw to final output
# =============================================================================
print("\n" + "=" * 80)
print("TEST 6: TRACE SINGLE HAND THROUGH PIPELINE")
print("=" * 80)

# Get a hand_id that exists in final output
try:
    final_df = spark.read.parquet(PATHS['sp7_output'])
    TRACE_HAND = final_df.select('hand_id').first()['hand_id']
    print(f"\nTracing hand: {TRACE_HAND}")
except:
    # Fall back to sp4 output
    try:
        final_df = spark.read.parquet(PATHS['sp4_output'])
        TRACE_HAND = final_df.select('hand_id').first()['hand_id']
        print(f"\nTracing hand: {TRACE_HAND} (from sp4)")
    except:
        TRACE_HAND = None
        print("Could not find a hand to trace")

if TRACE_HAND:
    for name, path in PATHS.items():
        print(f"\n--- {name} ---")
        try:
            df = spark.read.parquet(path)
            hand_df = df.filter(F.col('hand_id') == TRACE_HAND)
            count = hand_df.count()
            print(f"   Rows for this hand: {count}")
            
            if count > 0:
                # Determine order column
                order_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
                
                # Select key columns
                show_cols = ['hand_id', 'actor', 'street', 'action_type']
                if order_col:
                    show_cols.insert(1, order_col)
                if 'amount' in df.columns:
                    show_cols.append('amount')
                if 'facing_call' in df.columns:
                    show_cols.append('facing_call')
                
                show_cols = [c for c in show_cols if c in df.columns]
                
                if order_col:
                    hand_df.select(show_cols).orderBy(order_col).show(20, truncate=False)
                else:
                    hand_df.select(show_cols).show(20, truncate=False)
            else:
                print("   Hand not found in this dataset")
                
        except Exception as e:
            print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 7: COMPARE RAW ACTIONS TO SP-2 OUTPUT
# What's being lost in the first transformation?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 7: RAW ACTIONS vs SP-2 OUTPUT")
print("=" * 80)

try:
    raw = spark.read.parquet(PATHS['raw_actions'])
    sp2 = spark.read.parquet(PATHS['sp2_output'])
    
    print(f"\nRaw actions: {raw.count():,} rows")
    print(f"SP-2 output: {sp2.count():,} rows")
    print(f"Reduction: {100 - 100*sp2.count()/raw.count():.1f}%")
    
    print("\n--- Raw Actions Street Distribution ---")
    raw.groupBy('street').count().orderBy('count', ascending=False).show()
    
    print("\n--- SP-2 Output Street Distribution ---")
    sp2.groupBy('street').count().orderBy('count', ascending=False).show()
    
    # Check hand_id overlap
    raw_hands = raw.select('hand_id').distinct().count()
    sp2_hands = sp2.select('hand_id').distinct().count()
    print(f"\nUnique hands in raw: {raw_hands:,}")
    print(f"Unique hands in SP-2: {sp2_hands:,}")
    print(f"Hands retained: {100*sp2_hands/raw_hands:.1f}%")
    
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 8: STREET ORDERING VALIDATION
# Is idx ordered correctly? (preflop -> flop -> turn -> river)
# =============================================================================
print("\n" + "=" * 80)
print("TEST 8: STREET ORDERING VALIDATION")
print("=" * 80)

from pyspark.sql.window import Window

# Define correct street order
STREET_ORDER = {'preflop': 0, 'flop': 1, 'turn': 2, 'river': 3, 'showdown': 4}

for name, path in PATHS.items():
    if 'sp4' not in name and 'sp6' not in name and 'sp7' not in name:
        continue
        
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        
        idx_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
        
        if not idx_col:
            print("   No idx column found - skipping")
            continue
        
        # Map street to numeric order
        street_order_expr = F.create_map([F.lit(x) for item in STREET_ORDER.items() for x in item])
        df = df.withColumn('_street_order', street_order_expr[F.col('street')])
        
        # Check if idx ordering matches street ordering within each hand
        w = Window.partitionBy('hand_id').orderBy(idx_col)
        df = df.withColumn('_prev_street_order', F.lag('_street_order').over(w))
        
        # Street should never go backwards (e.g., flop -> preflop is wrong)
        backwards = df.filter(
            (F.col('_prev_street_order').isNotNull()) &
            (F.col('_street_order') < F.col('_prev_street_order'))
        ).count()
        
        print(f"   Actions where street goes BACKWARDS: {backwards}")
        
        if backwards > 0:
            print("   *** CRITICAL: Street order is WRONG - actions out of sequence! ***")
            df.filter(
                (F.col('_prev_street_order').isNotNull()) &
                (F.col('_street_order') < F.col('_prev_street_order'))
            ).select('hand_id', idx_col, 'street', '_street_order', '_prev_street_order').show(5)
        else:
            print("   ✓ Street ordering is correct")
        
        # Also check first action per hand should be preflop
        first_actions = df.groupBy('hand_id').agg(F.min(idx_col).alias('first_idx'))
        df_with_first = df.join(first_actions, on='hand_id')
        first_not_preflop = df_with_first.filter(
            (F.col(idx_col) == F.col('first_idx')) &
            (F.col('street') != 'preflop')
        ).count()
        
        print(f"   Hands where first action is NOT preflop: {first_not_preflop}")
        if first_not_preflop > 0:
            print("   *** WARNING: Some hands don't start with preflop ***")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 9: POT PROGRESSION VALIDATION
# Does pot grow correctly through the hand?
# =============================================================================
print("\n" + "=" * 80)
print("TEST 9: POT PROGRESSION VALIDATION")
print("=" * 80)

for name, path in PATHS.items():
    if 'sp4' not in name and 'sp6' not in name and 'sp7' not in name:
        continue
        
    print(f"\n--- {name} ---")
    try:
        df = spark.read.parquet(path)
        
        idx_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
        
        if 'pot_before_action' not in df.columns:
            print("   No pot_before_action column - skipping")
            continue
            
        if not idx_col:
            print("   No idx column - skipping")
            continue
        
        # Check 1: pot_before_action should never be negative
        negative_pots = df.filter(F.col('pot_before_action') < 0).count()
        print(f"   Negative pot_before_action: {negative_pots}")
        if negative_pots > 0:
            print("   *** WARNING: Negative pot values! ***")
        
        # Check 2: pot should generally grow (or stay same) within a hand
        w = Window.partitionBy('hand_id').orderBy(idx_col)
        df = df.withColumn('_prev_pot', F.lag('pot_before_action').over(w))
        
        # Pot going down could indicate data issues (except at start of new street)
        pot_decreased = df.filter(
            (F.col('_prev_pot').isNotNull()) &
            (F.col('pot_before_action') < F.col('_prev_pot'))
        ).count()
        
        print(f"   Actions where pot DECREASED: {pot_decreased}")
        if pot_decreased > 0:
            print("   Note: Some decrease is normal at street transitions in some data formats")
        
        # Check 3: First action of hand should have small pot (blinds only)
        first_actions = df.groupBy('hand_id').agg(F.min(idx_col).alias('first_idx'))
        df_first = df.join(first_actions, on='hand_id').filter(F.col(idx_col) == F.col('first_idx'))
        
        # First action pot should be 0 or small (blinds)
        first_pot_zero = df_first.filter(F.col('pot_before_action') == 0).count()
        first_pot_small = df_first.filter(F.col('pot_before_action') <= 3).count()  # Assume 1BB + 0.5SB = 1.5
        total_first = df_first.count()
        
        print(f"   First actions with pot=0: {first_pot_zero}/{total_first}")
        print(f"   First actions with pot<=3: {first_pot_small}/{total_first}")
        
        if first_pot_zero < total_first * 0.5:
            print("   Note: First action usually has blinds posted already")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 10: FULL HAND TRACE WITH VALIDATION
# Pick a hand with all streets and validate the entire sequence
# =============================================================================
print("\n" + "=" * 80)
print("TEST 10: FULL HAND TRACE WITH VALIDATION")
print("=" * 80)

for name, path in PATHS.items():
    if 'sp4' not in name and 'sp7' not in name:
        continue
        
    print(f"\n{'='*40}")
    print(f"--- {name} ---")
    print(f"{'='*40}")
    
    try:
        df = spark.read.parquet(path)
        
        idx_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
        
        if not idx_col:
            print("   No idx column - skipping")
            continue
        
        # Find a hand with all 4 streets
        hand_streets = df.groupBy('hand_id').agg(
            F.countDistinct('street').alias('num_streets'),
            F.count('*').alias('num_actions')
        )
        full_hands = hand_streets.filter(F.col('num_streets') >= 4).orderBy(F.col('num_actions').desc())
        
        if full_hands.count() == 0:
            print("   No hands with all 4 streets found")
            continue
        
        trace_hand = full_hands.first()['hand_id']
        print(f"\n   Tracing hand: {trace_hand}")
        
        # Get all actions for this hand
        hand_df = df.filter(F.col('hand_id') == trace_hand).orderBy(idx_col)
        
        # Build trace columns
        trace_cols = [idx_col, 'actor', 'street', 'action_type']
        for col in ['amount', 'facing_call', 'pot_before_action', 'starting_stack']:
            if col in df.columns:
                trace_cols.append(col)
        
        print(f"\n   Action sequence:")
        hand_df.select(trace_cols).show(30, truncate=False)
        
        # Validation checks for this specific hand
        print("\n   Hand-level validation:")
        
        actions = hand_df.collect()
        
        prev_street_order = -1
        prev_actor = None
        issues = []
        
        street_map = {'preflop': 0, 'flop': 1, 'turn': 2, 'river': 3}
        
        for i, row in enumerate(actions):
            street = row['street']
            actor = row['actor']
            action_type = row['action_type']
            idx_val = row[idx_col]
            
            street_order = street_map.get(street, 99)
            
            # Check 1: Street should not go backwards
            if street_order < prev_street_order:
                issues.append(f"   idx={idx_val}: Street went BACKWARDS ({street} after previous)")
            
            # Check 2: Same player shouldn't act twice in a row (within same street)
            if actor == prev_actor and street_order == prev_street_order:
                issues.append(f"   idx={idx_val}: Same player ({actor[:10]}...) acting twice consecutively")
            
            prev_street_order = street_order
            prev_actor = actor
        
        if issues:
            print(f"   *** ISSUES FOUND: {len(issues)} ***")
            for issue in issues[:10]:  # Show first 10
                print(issue)
        else:
            print("   ✓ Hand sequence looks valid")
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# TEST 11: DEEP DIVE - BETTING WHEN FACING SHOVE
# Find specific examples and trace back to raw data to understand root cause
# =============================================================================
print("\n" + "=" * 80)
print("TEST 11: DEEP DIVE - BETTING WHEN FACING SHOVE")
print("=" * 80)

from pyspark.sql.window import Window

# Check processed outputs
for name, path in [('sp4_output', PATHS['sp4_output']), ('sp7_output', PATHS['sp7_output'])]:
    print(f"\n{'='*60}")
    print(f"--- {name} ---")
    print(f"{'='*60}")
    
    try:
        df = spark.read.parquet(path)
        
        idx_col = 'idx' if 'idx' in df.columns else 'action_no_in_hand' if 'action_no_in_hand' in df.columns else None
        
        if not idx_col or 'amount' not in df.columns:
            print("   Missing required columns - skipping")
            continue
        
        # Window to get previous action info
        w = Window.partitionBy('hand_id').orderBy(idx_col)
        
        # Add context columns
        df_context = df.withColumn('prev_amount', F.lag('amount').over(w)) \
                       .withColumn('prev_action', F.lag('action_type').over(w)) \
                       .withColumn('prev_actor', F.lag('actor').over(w)) \
                       .withColumn('prev_street', F.lag('street').over(w)) \
                       .withColumn('prev_idx', F.lag(idx_col).over(w))
        
        # Add starting_stack for all-in detection
        if 'starting_stack' in df.columns:
            df_context = df_context.withColumn('prev_stack', F.lag('starting_stack').over(w))
            
            # Case 1: Previous player went all-in, current player raises (not just calls)
            # This is ONLY valid if current player has more chips and is re-raising
            # But it's INVALID if shown as a smaller raise amount
            
            # Find cases where:
            # - Previous action was a bet/raise for >= 90% of their stack (all-in or near all-in)
            # - Current action is also bet/raise
            # - Current amount is LESS than previous amount (impossible - must call first)
            
            suspicious = df_context.filter(
                (F.col('prev_action') == 'bet_or_raise_to') &
                (F.col('prev_amount') >= F.col('prev_stack') * 0.9) &  # Previous was all-in
                (F.col('action_type') == 'bet_or_raise_to') &
                (F.col('amount') < F.col('prev_amount')) &  # Current amount is LESS (impossible)
                (F.col('actor') != F.col('prev_actor'))  # Different player
            )
            
            suspicious_count = suspicious.count()
            print(f"\n   SUSPICIOUS: Raise LESS than facing all-in: {suspicious_count}")
            
            if suspicious_count > 0:
                print("   *** CRITICAL: Player 'raising' to less than the all-in amount! ***")
                print("   This is impossible in poker - must at least call the all-in\n")
                
                # Show examples
                suspicious.select(
                    'hand_id', idx_col, 'actor', 'street', 'action_type', 'amount',
                    'prev_idx', 'prev_actor', 'prev_action', 'prev_amount', 'prev_stack'
                ).show(10, truncate=False)
                
                # Trace ONE problematic hand back to raw data
                problem_hand = suspicious.select('hand_id').first()['hand_id']
                print(f"\n   --- TRACING PROBLEM HAND: {problem_hand} ---")
                
                # Get full action sequence for this hand from current output
                print(f"\n   Actions in {name}:")
                df.filter(F.col('hand_id') == problem_hand).orderBy(idx_col).select(
                    idx_col, 'actor', 'street', 'action_type', 'amount', 'starting_stack'
                ).show(20, truncate=False)
                
                # Compare to raw data
                try:
                    raw = spark.read.parquet(PATHS['raw_actions'])
                    print(f"\n   Actions in RAW DATA:")
                    raw.filter(F.col('hand_id') == problem_hand).orderBy('idx').select(
                        'idx', 'actor', 'street', 'action_type', 'amount'
                    ).show(20, truncate=False)
                    
                    # Check if ordering matches
                    raw_order = [r['idx'] for r in raw.filter(F.col('hand_id') == problem_hand).orderBy('idx').collect()]
                    proc_order = [r[idx_col] for r in df.filter(F.col('hand_id') == problem_hand).orderBy(idx_col).collect()]
                    
                    print(f"\n   Raw idx sequence: {raw_order[:15]}...")
                    print(f"   Processed idx sequence: {proc_order[:15]}...")
                    
                    if raw_order == proc_order[:len(raw_order)]:
                        print("   ✓ Ordering MATCHES raw data")
                        print("   => Issue may be in the raw data itself or amount calculation")
                    else:
                        print("   *** MISMATCH: Ordering differs from raw data! ***")
                        print("   => Issue is in the pipeline ordering")
                        
                except Exception as e:
                    print(f"   Could not read raw data: {e}")
            else:
                print("   ✓ No impossible raise amounts found")
        
        # Case 2: Same player betting twice in same street (regardless of all-in)
        same_player_twice = df_context.filter(
            (F.col('actor') == F.col('prev_actor')) &
            (F.col('street') == F.col('prev_street')) &
            (F.col('action_type') == 'bet_or_raise_to') &
            (F.col('prev_action') == 'bet_or_raise_to')
        ).count()
        
        print(f"\n   Same player raising twice consecutively in same street: {same_player_twice}")
        if same_player_twice > 0:
            print("   *** CRITICAL: Player raising twice in a row (impossible!) ***")
            df_context.filter(
                (F.col('actor') == F.col('prev_actor')) &
                (F.col('street') == F.col('prev_street')) &
                (F.col('action_type') == 'bet_or_raise_to') &
                (F.col('prev_action') == 'bet_or_raise_to')
            ).select(
                'hand_id', idx_col, 'actor', 'street', 'action_type', 'amount',
                'prev_idx', 'prev_action', 'prev_amount'
            ).show(5, truncate=False)
            
    except Exception as e:
        print(f"   ERROR: {e}")

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

print("""
Review the output above to identify:

TEST 1: Street Distribution
  - All streets should be present (preflop, flop, turn, river)
  - Major reduction in one street indicates filtering issue

TEST 2: Duplicate Detection  
  - Zero duplicates expected
  - Duplicates indicate JOIN issues in pipeline

TEST 3: Action Type Distribution
  - Folds should be ~30-50% of actions (most common)
  - Zero folds indicates showdown-only filter removing folders

TEST 4: Column Inventory
  - Critical columns should be present at each stage
  - Missing columns indicate transformation issues

TEST 5: Betting Sanity
  - Zero impossible raises (amount < facing_call)
  - Zero consecutive same-player actions
  - Zero betting after all-in

TEST 6: Hand Trace
  - Actions should progress logically through streets
  - Players should alternate turns

TEST 7: Raw vs SP-2 Comparison
  - Expected: significant reduction due to showdown filter
  - All streets should still be proportionally represented

TEST 8: Street Ordering
  - Zero backwards street transitions (e.g., flop → preflop)
  - First action should always be preflop

TEST 9: Pot Progression
  - Pot should generally increase through hand
  - First action should have small/zero pot

TEST 10: Full Hand Trace
  - Visual inspection of one complete hand
  - Validates entire action sequence

TEST 11: Betting When Facing Shove (NEW)
  - Zero cases of raising to LESS than the all-in amount
  - Zero cases of same player raising twice consecutively
  - Traces problematic hands back to raw data to identify root cause

ROOT CAUSE ANALYSIS:
If Test 11 finds issues and ordering MATCHES raw data:
  => Issue is in the raw data or amount calculation (not ordering)
  
If Test 11 finds issues and ordering DIFFERS from raw data:
  => Issue is in the pipeline ordering (check SP-2 cell-9 fix)

FIX ISSUES IN ORDER:
1. SP-2 issues (missing columns, filtering, ordering)
2. SP-4 issues (idx ordering, JOIN duplicates)  
3. SP-6/SP-7 issues (derived from upstream)
""")